In [1]:
import pandas as pd
import numpy as np
import itertools
import pandas_gbq
import datetime
import matplotlib.pyplot as plt
from datetime import *
from datetime import datetime, timedelta, date
from pathlib import Path
from PIL import Image
# %load_ext google.colab.data_table
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
project_id = "perceptive-ivy-290216"

# Standard plotly imports
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
query2=f"""
SELECT
*
FROM `perceptive-ivy-290216.f1_api.sprint_lap_time`  A
# WHERE A.Year=2024
# AND A.GP="Monaco Grand Prix"
# AND A.DRIVER='HAM'
ORDER BY LapNumber, LapStartTime
"""
track3=pandas_gbq.read_gbq(query2,project_id,dialect='standard')

Downloading: 100%|██████████|


In [4]:
track2=track3[(track3["GP"]=='Chinese Grand Prix')&(track3["Year"]==2025)]

In [5]:
track2.head()
year=track2['Year'].iloc[0]
gp=track2['GP'].iloc[0]

In [6]:
track2['LapTime']= pd.to_timedelta(track2["LapTime"])

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_43972/4173085553.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  track2['LapTime']= pd.to_timedelta(track2["LapTime"])


In [7]:
# track2['LapTime'].dt.total_seconds().min()*1.07
track2=track2[track2['LapNumber']!=1.0]
track2.loc[:, "LapTime (s)"] = track2["LapTime"].dt.total_seconds()

In [8]:
# quicklaps=track2[track2["LapTime"]<track2['LapTime'].min()*1.35]
quicklaps=track2
quicklaps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Year,GP,LapTime (s)
511,0 days 00:48:25.746000,HAM,44,0 days 00:01:35.399000,2.0,1.0,NaT,NaT,0 days 00:00:25.591000,0 days 00:00:28.731000,0 days 00:00:41.077000,0 days 00:47:16.014000,0 days 00:47:44.745000,0 days 00:48:25.822000,NaN,275.0,260.0,312.0,True,MEDIUM,8.0,False,Ferrari,0 days 00:46:50.347000,2025-03-22 03:05:21.513,1,1.0,False,,False,True,2025,Chinese Grand Prix,95.399
512,0 days 00:48:26.863000,VER,1,0 days 00:01:35.745000,2.0,1.0,NaT,NaT,0 days 00:00:25.654000,0 days 00:00:28.886000,0 days 00:00:41.205000,0 days 00:47:16.833000,0 days 00:47:45.719000,0 days 00:48:26.924000,279.0,275.0,261.0,317.0,True,MEDIUM,5.0,False,Red Bull Racing,0 days 00:46:51.118000,2025-03-22 03:05:22.284,1,2.0,False,,False,True,2025,Chinese Grand Prix,95.745
513,0 days 00:48:28.523000,PIA,81,0 days 00:01:36.086000,2.0,1.0,NaT,NaT,0 days 00:00:25.510000,0 days 00:00:29.217000,0 days 00:00:41.359000,0 days 00:47:18.038000,0 days 00:47:47.255000,0 days 00:48:28.614000,273.0,275.0,261.0,315.0,True,MEDIUM,6.0,False,McLaren,0 days 00:46:52.437000,2025-03-22 03:05:23.603,1,3.0,False,,False,True,2025,Chinese Grand Prix,96.086
515,0 days 00:48:30.092000,RUS,63,0 days 00:01:36.432000,2.0,1.0,NaT,NaT,0 days 00:00:26.032000,0 days 00:00:29.221000,0 days 00:00:41.179000,0 days 00:47:19.692000,0 days 00:47:48.913000,0 days 00:48:30.092000,275.0,272.0,262.0,319.0,True,MEDIUM,10.0,False,Mercedes,0 days 00:46:53.660000,2025-03-22 03:05:24.826,1,4.0,False,,False,True,2025,Chinese Grand Prix,96.432
516,0 days 00:48:30.902000,LEC,16,0 days 00:01:36.895000,2.0,1.0,NaT,NaT,0 days 00:00:26.226000,0 days 00:00:29.410000,0 days 00:00:41.259000,0 days 00:47:20.278000,0 days 00:47:49.688000,0 days 00:48:30.947000,284.0,279.0,268.0,337.0,True,MEDIUM,8.0,False,Ferrari,0 days 00:46:54.007000,2025-03-22 03:05:25.173,1,5.0,False,,False,True,2025,Chinese Grand Prix,96.895


In [9]:
#Remove Pitstops and Track Status other than Clear to remove slow laps
quicklaps=quicklaps[((quicklaps["PitOutTime"]=='NaT')&(quicklaps["PitInTime"]=='NaT')&(~quicklaps["TrackStatus"].isin(['4','41','5','6','7','124'])))]

In [10]:
transformed_laps = quicklaps.copy()
transformed_laps.loc[:, "LapTime (s)"] = quicklaps["LapTime"].dt.total_seconds()

# order the team from the fastest (lowest median lap time) to slower
team_order = (
    transformed_laps[["Team", "LapTime (s)"]].groupby("Team").median()["LapTime (s)"].sort_values().index
)
print(team_order)

Index(['Ferrari', 'McLaren', 'Mercedes', 'Williams', 'Aston Martin',
       'Red Bull Racing', 'Racing Bulls', 'Alpine', 'Haas F1 Team',
       'Kick Sauber'],
      dtype='object', name='Team')


In [11]:
transformed_laps['median'] = transformed_laps['LapTime (s)'].groupby(transformed_laps['Team']).transform('median')
transformed_laps.tail()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Year,GP,LapTime (s),median
6603,0 days 01:16:47.619000,BOR,5,0 days 00:01:48.294000,19.0,1.0,NaT,NaT,0 days 00:00:26.399000,0 days 00:00:31.989000,0 days 00:00:49.906000,0 days 01:15:25.774000,0 days 01:15:57.763000,0 days 01:16:47.669000,288.0,276.0,250.0,338.0,False,MEDIUM,25.0,False,Kick Sauber,0 days 01:14:59.325000,2025-03-22 03:33:30.491,12,18.0,False,,False,True,2025,Chinese Grand Prix,108.294,99.097
6604,0 days 01:16:39.861000,OCO,31,0 days 00:01:39.916000,19.0,1.0,NaT,NaT,0 days 00:00:26.299000,0 days 00:00:30.700000,0 days 00:00:42.917000,0 days 01:15:26.314000,0 days 01:15:57.014000,0 days 01:16:39.931000,NaN,275.0,266.0,341.0,False,MEDIUM,25.0,False,Haas F1 Team,0 days 01:14:59.945000,2025-03-22 03:33:31.111,1,16.0,False,,False,True,2025,Chinese Grand Prix,99.916,98.725
6605,0 days 01:16:53.891000,DOO,7,0 days 00:01:52.236000,19.0,1.0,NaT,NaT,0 days 00:00:26.524000,0 days 00:00:30.024000,0 days 00:00:55.688000,0 days 01:15:28.282000,0 days 01:15:58.306000,0 days 01:16:53.994000,279.0,274.0,260.0,346.0,False,MEDIUM,25.0,False,Alpine,0 days 01:15:01.655000,2025-03-22 03:33:32.821,12,20.0,False,,False,True,2025,Chinese Grand Prix,112.236,98.657
6606,0 days 01:16:50.361000,HUL,27,0 days 00:01:46.846000,19.0,1.0,NaT,NaT,0 days 00:00:26.367000,0 days 00:00:29.783000,0 days 00:00:50.696000,0 days 01:15:29.934000,0 days 01:15:59.717000,0 days 01:16:50.413000,NaN,270.0,187.0,313.0,False,MEDIUM,25.0,False,Kick Sauber,0 days 01:15:03.515000,2025-03-22 03:33:34.681,12,19.0,False,,False,True,2025,Chinese Grand Prix,106.846,99.097
6607,0 days 01:16:45.055000,SAI,55,0 days 00:01:39.421000,19.0,2.0,NaT,NaT,0 days 00:00:26.078000,0 days 00:00:29.368000,0 days 00:00:43.975000,0 days 01:15:31.764000,0 days 01:16:01.132000,0 days 01:16:45.107000,281.0,277.0,261.0,322.0,False,MEDIUM,14.0,False,Williams,0 days 01:15:05.634000,2025-03-22 03:33:36.800,1,17.0,False,,False,True,2025,Chinese Grand Prix,99.421,97.915


In [12]:
fig_pace=px.box(
    transformed_laps.sort_values(by=["median","LapNumber"]),
    x="Team",
    y="LapTime (s)",
    color='Team',
    template="xgridoff",
    title="<b>Team Pace Plot for the {} {} Sprint Race</b>".format(year,gp),
    # palette=team_palette,
    # whiskerprops=dict(color="white"),
    # boxprops=dict(edgecolor="white"),
    # medianprops=dict(color="grey"),
    # capprops=dict(color="white"),
    height=700, width=1200,

color_discrete_map={
                 "Alpine": "#0093cc",
                 "Aston Martin": "#229971",
                 "Ferrari": "#E80020",
                 "Haas F1 Team": "#B6BABD",
                 "Kick Sauber": "#52e252",
                 "McLaren": "#FF8000",
                 "Mercedes": "#27F4D2",
                 "RB": "#6692FF",
                 "Racing Bulls": "#6692FF",
                 "Red Bull Racing": "#3671C6",
                 "Williams": "#64C4FF" ,
                 "Alfa Romeo":"#C92D4B",
                 "AlphaTauri":"#5E8FAA",
                 "Racing Point":"#F596C8",
                 "Renault":"#FFF500",
                 "Toro Rosso":"#469bff",
                 "Force India":"#F596C8",
                 "Sauber":"#9B0000"
                 }
)
fig_pace.update_traces(opacity=1)
fig_pace.update_layout(
    hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
    ),
    title_x=0.5,
    margin=dict(l=60, r=5, t=50, b=60),
    yaxis = dict(tickfont = dict(size=15)),
    xaxis = dict(tickfont = dict(size=15)),
    font=dict(
        family="PT Sans Narrow",
        size=14,
        color="Black"
    ),
    title_font_family="PT Sans Narrow"
)
fig_pace.show()

In [13]:
fig_pace.write_html("/Users/rdesh723/statpulse-html/plots/Sprint Race Team Pace/Team Pace Plots/{}/{} {} Sprint Race Team Pace Plot.html".format(year,year,gp),full_html=False, include_plotlyjs='cdn')

In [14]:
#Get MINIMUM OF THE MEDIAN LAP TIMES

fastest_lap = transformed_laps["median"].min()
transformed_laps['Delta']= transformed_laps["median"] - (fastest_lap)
transformed_laps['Delta_Percent']= (transformed_laps["median"]/(fastest_lap)-1)*100
transformed_laps['Delta_Percent_Abs']=transformed_laps['Delta_Percent']
transformed_laps['Delta_Percent']=transformed_laps['Delta_Percent']+100
transformed_laps['Delta_Percent']=transformed_laps['Delta_Percent'].round(3)
transformed_laps['Delta_Percent']=transformed_laps['Delta_Percent'].astype(str)
transformed_laps['Delta_Percent']=transformed_laps['Delta_Percent']+"%"
transformed_laps_relative_median=transformed_laps[["Team","Delta","Delta_Percent","Delta_Percent_Abs"]].drop_duplicates()
transformed_laps_relative_median=transformed_laps_relative_median.sort_values("Delta")
transformed_laps_relative_median

,Team,Delta,Delta_Percent,Delta_Percent_Abs
511,Ferrari,0.0000,100.0%,0.000000
513,McLaren,0.3585,100.369%,0.369310
515,Mercedes,0.4920,100.507%,0.506835
525,Williams,0.8420,100.867%,0.867388
520,Aston Martin,0.9785,101.008%,1.008004
512,Red Bull Racing,1.0200,101.051%,1.050756
517,Racing Bulls,1.2060,101.242%,1.242364
528,Alpine,1.5840,101.632%,1.631762
526,Haas F1 Team,1.6520,101.702%,1.701812
530,Kick Sauber,2.0240,102.085%,2.085029


In [15]:
for index, row in transformed_laps_relative_median.iterrows():
  if transformed_laps_relative_median.loc[index, 'Delta']==0.0:
    transformed_laps_relative_median.loc[index, 'Delta']=0.001
for index, row in transformed_laps_relative_median.iterrows():
  if transformed_laps_relative_median.loc[index, 'Delta_Percent_Abs']==0.0:
    transformed_laps_relative_median.loc[index, 'Delta_Percent_Abs']=0.001
        
transformed_laps_relative_median

,Team,Delta,Delta_Percent,Delta_Percent_Abs
511,Ferrari,0.0010,100.0%,0.001000
513,McLaren,0.3585,100.369%,0.369310
515,Mercedes,0.4920,100.507%,0.506835
525,Williams,0.8420,100.867%,0.867388
520,Aston Martin,0.9785,101.008%,1.008004
512,Red Bull Racing,1.0200,101.051%,1.050756
517,Racing Bulls,1.2060,101.242%,1.242364
528,Alpine,1.5840,101.632%,1.631762
526,Haas F1 Team,1.6520,101.702%,1.701812
530,Kick Sauber,2.0240,102.085%,2.085029


In [16]:
fig_median=px.bar(
    transformed_laps_relative_median,
    x="Team",
    y="Delta_Percent_Abs",
    color='Team',
    template="xgridoff",
    title="<b>Team Pace (Median Lap Times) for the {} {} Sprint Race</b>".format(year,gp),
    text="Delta_Percent",
    height=700, width=1200,
color_discrete_map={
                 "Alpine": "#0093cc",
                 "Aston Martin": "#229971",
                 "Ferrari": "#E80020",
                 "Haas F1 Team": "#B6BABD",
                 "Kick Sauber": "#52e252",
                 "McLaren": "#FF8000",
                 "Mercedes": "#27F4D2",
                 "RB": "#6692FF",
                 "Red Bull Racing": "#3671C6",
                 "Williams": "#64C4FF" ,
                 "Alfa Romeo":"#C92D4B",
                 "AlphaTauri":"#5E8FAA",
                 "Racing Point":"#F596C8",
                 "Renault":"#FFF500",
                 "Toro Rosso":"#469bff",
                 "Force India":"#F596C8",
                 "Sauber":"#9B0000"
                 }
)

fig_median.update_traces(textposition='outside')

fig_median.update_traces(opacity=0.85)

fig_median.update_traces( marker_line_color='white',marker_line_width=1.5)

fig_median.update_yaxes(tickangle = -90,tickprefix="<b>",ticksuffix ="</b><br>")

fig_median.update_xaxes(tickangle = -0,tickprefix="<b>",ticksuffix ="</b><br>")

fig_median.update_yaxes(range=[transformed_laps_relative_median['Delta_Percent_Abs'].max()+0.8, 0])

# fig['layout']['yaxis']['autorange'] = "reversed"

fig_median.update_layout(
    xaxis={'side': 'top'},
    yaxis={'side': 'left'}
)

fig_median.update_xaxes(title_text='')      # Remove axis titles
fig_median.update_yaxes(title_text='Percentage%')

fig_median.update_xaxes(
        title_standoff = 75
)

for x,y in zip(transformed_laps_relative_median.Team, transformed_laps_relative_median.Delta_Percent_Abs):
  for png in (Path(Path.cwd()).parents[0].joinpath("F1_LOGOS").glob("*.png")):
    if str.split(str.split(str(png),".")[0],"/")[6]==x:
      image=str(png)
      fig_median.add_layout_image(
          x=x,
          y=y+0.3,
          source=Image.open(image),
          xref="x",
          yref="y",
          sizex=0.9,
          sizey=0.9,
          xanchor="center",
          yanchor="middle",
      )

fig_median.update_layout(
    title_x=0.5,
    margin=dict(l=100, r=5, t=110, b=30),
    hoverlabel=dict(
    bgcolor="white",
    font_size=20,
    font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=15)),
    xaxis = dict(tickfont = dict(size=15)),
    font=dict(
        family="PT Sans Narrow",
        size=16,
        color="Black"
    ),
    title_font_family="PT Sans Narrow"
)
fig_median.update_traces( marker_line_color='Black',marker_line_width=0.1, opacity=0.99)

fig_median.show()

In [17]:
fig_median.write_html("/Users/rdesh723/statpulse-html/plots/Sprint Race Team Pace/Median Lap Times/{}/{} {} Sprint Race Team Median Plot.html".format(year,year,gp),full_html=False, include_plotlyjs='cdn')

In [18]:
#Get MINIMUM OF ALL LAP TIMES
transformed_laps['min_all'] = transformed_laps['LapTime (s)'].groupby(transformed_laps['Team']).transform('min')

fastest_lap_all = transformed_laps["min_all"].min()
transformed_laps['Delta_Min_All']= transformed_laps["min_all"] - (fastest_lap_all)
transformed_laps['Delta_Percent_All']= (transformed_laps["min_all"]/(fastest_lap_all)-1)*100
transformed_laps['Delta_Percent_Abs']=transformed_laps['Delta_Percent_All']
transformed_laps['Delta_Percent_All']=transformed_laps['Delta_Percent_All']+100
transformed_laps['Delta_Percent_All']=transformed_laps['Delta_Percent_All'].round(3)
transformed_laps['Delta_Percent_All']=transformed_laps['Delta_Percent_All'].astype(str)
transformed_laps['Delta_Percent_All']=transformed_laps['Delta_Percent_All']+"%"
transformed_laps_relative_min_all=transformed_laps[["Team","Delta_Min_All","Delta_Percent_All","Delta_Percent_Abs"]].drop_duplicates()
transformed_laps_relative_min_all=transformed_laps_relative_min_all.sort_values("Delta_Min_All")
transformed_laps_relative_min_all

,Team,Delta_Min_All,Delta_Percent_All,Delta_Percent_Abs
511,Ferrari,0.000,100.0%,0.000000
512,Red Bull Racing,0.346,100.363%,0.362687
525,Williams,0.420,100.44%,0.440256
513,McLaren,0.455,100.477%,0.476944
515,Mercedes,0.492,100.516%,0.515729
517,Racing Bulls,0.989,101.037%,1.036698
520,Aston Martin,1.036,101.086%,1.085965
530,Kick Sauber,1.130,101.184%,1.184499
526,Haas F1 Team,1.736,101.82%,1.819726
528,Alpine,2.082,102.182%,2.182413


In [19]:
for index, row in transformed_laps_relative_min_all.iterrows():
  if transformed_laps_relative_min_all.loc[index, 'Delta_Min_All']==0.0:
    transformed_laps_relative_min_all.loc[index, 'Delta_Min_All']=0.001
for index, row in transformed_laps_relative_min_all.iterrows():
  if transformed_laps_relative_min_all.loc[index, 'Delta_Percent_Abs']==0.0:
    transformed_laps_relative_min_all.loc[index, 'Delta_Percent_Abs']=0.001
        
transformed_laps_relative_min_all

,Team,Delta_Min_All,Delta_Percent_All,Delta_Percent_Abs
511,Ferrari,0.001,100.0%,0.001000
512,Red Bull Racing,0.346,100.363%,0.362687
525,Williams,0.420,100.44%,0.440256
513,McLaren,0.455,100.477%,0.476944
515,Mercedes,0.492,100.516%,0.515729
517,Racing Bulls,0.989,101.037%,1.036698
520,Aston Martin,1.036,101.086%,1.085965
530,Kick Sauber,1.130,101.184%,1.184499
526,Haas F1 Team,1.736,101.82%,1.819726
528,Alpine,2.082,102.182%,2.182413


In [20]:
fig_min=px.bar(
    transformed_laps_relative_min_all,
    x="Team",
    y="Delta_Percent_Abs",
    color='Team',
    template="xgridoff",
    title="<b>Team Pace (Fastest Lap Times) for the {} {} Sprint Race</b>".format(year,gp),
    text="Delta_Percent_All",
    height=700, width=1200,
color_discrete_map={
                 "Alpine": "#0093cc",
                 "Aston Martin": "#229971",
                 "Ferrari": "#E80020",
                 "Haas F1 Team": "#B6BABD",
                 "Kick Sauber": "#52e252",
                 "McLaren": "#FF8000",
                 "Mercedes": "#27F4D2",
                 "RB": "#6692FF",
                 "Racing Bulls": "#6692FF",
                 "Red Bull Racing": "#3671C6",
                 "Williams": "#64C4FF" ,
                 "Alfa Romeo":"#C92D4B",
                 "AlphaTauri":"#5E8FAA",
                 "Racing Point":"#F596C8",
                 "Renault":"#FFF500",
                 "Toro Rosso":"#469bff",
                 "Force India":"#F596C8",
                 "Sauber":"#9B0000"
                 }
)

fig_min.update_traces(textposition='outside')

fig_min.update_traces(opacity=0.85)

fig_min.update_traces( marker_line_color='white',marker_line_width=1.5)

fig_min.update_yaxes(tickangle = -90,tickprefix="<b>",ticksuffix ="</b><br>")

fig_min.update_xaxes(tickangle = -0,tickprefix="<b>",ticksuffix ="</b><br>")

fig_min.update_yaxes(range=[transformed_laps_relative_min_all['Delta_Percent_Abs'].max()+0.8, 0])

# fig['layout']['yaxis']['autorange'] = "reversed"

fig_min.update_layout(
    xaxis={'side': 'top'},
    yaxis={'side': 'left'}
)

fig_min.update_xaxes(title_text='')      # Remove axis titles
fig_min.update_yaxes(title_text='Percentage%')

fig_min.update_xaxes(
        title_standoff = 75
)

for x,y in zip(transformed_laps_relative_min_all.Team, transformed_laps_relative_min_all.Delta_Percent_Abs):
  for png in (Path(Path.cwd()).parents[0].joinpath("F1_LOGOS").glob("*.png")):
    if str.split(str.split(str(png),".")[0],"/")[6]==x:
      image=str(png)
      fig_min.add_layout_image(
          x=x,
          y=y+0.4,
          source=Image.open(image),
          xref="x",
          yref="y",
          sizex=0.9,
          sizey=0.9,
          xanchor="center",
          yanchor="middle",
      )


fig_min.update_layout(
    title_x=0.5,
    margin=dict(l=100, r=5, t=110, b=30),
    hoverlabel=dict(
    bgcolor="white",
    font_size=20,
    font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=15)),
    xaxis = dict(tickfont = dict(size=15)),
    font=dict(
        family="PT Sans Narrow",
        size=16,
        color="Black"
    ),
    title_font_family="PT Sans Narrow"
)
fig_min.update_yaxes(ticksuffix = "  ")
fig_min.update_traces( marker_line_color='Black',marker_line_width=0.1, opacity=0.99)

fig_min.show()

In [21]:
fig_min.write_html("/Users/rdesh723/statpulse-html/plots/Sprint Race Team Pace/Fastest Lap/{}/{} {} Sprint Race Team Fastest Lap Plot.html".format(year,year,gp),full_html=False, include_plotlyjs='cdn')

In [22]:
#Get AVERAGE OF ALL LAP TIMES
transformed_laps['avg_all'] = transformed_laps['LapTime (s)'].groupby(transformed_laps['Team']).transform(np.mean)

fastest_lap_avg = transformed_laps["avg_all"].min()
transformed_laps['Delta_Avg_All']= transformed_laps["avg_all"] - (fastest_lap_avg)
transformed_laps['Delta_Percent_Avg']= (transformed_laps["avg_all"]/(fastest_lap_avg)-1)*100
transformed_laps['Delta_Percent_Abs']=transformed_laps['Delta_Percent_Avg']
transformed_laps['Delta_Percent_Avg']=transformed_laps['Delta_Percent_Avg']+100
transformed_laps['Delta_Percent_Avg']=transformed_laps['Delta_Percent_Avg'].round(3)
transformed_laps['Delta_Percent_Avg']=transformed_laps['Delta_Percent_Avg'].astype(str)
transformed_laps['Delta_Percent_Avg']=transformed_laps['Delta_Percent_Avg']+"%"
transformed_laps_relative_avg_all=transformed_laps[["Team","Delta_Avg_All","Delta_Percent_Avg","Delta_Percent_Abs"]].drop_duplicates()
transformed_laps_relative_avg_all=transformed_laps_relative_avg_all.sort_values("Delta_Avg_All")
transformed_laps_relative_avg_all

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_43972/970362312.py:2: FutureWarning:

The provided callable <function mean at 0x120beb4c0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.



,Team,Delta_Avg_All,Delta_Percent_Avg,Delta_Percent_Abs
511,Ferrari,0.000000,100.0%,0.000000
513,McLaren,0.378806,100.39%,0.390164
515,Mercedes,0.497917,100.513%,0.512847
512,Red Bull Racing,0.961167,100.99%,0.989988
517,Racing Bulls,1.175389,101.211%,1.210634
520,Aston Martin,1.179944,101.215%,1.215326
525,Williams,1.201631,101.238%,1.237663
526,Haas F1 Team,1.837583,101.893%,1.892685
528,Alpine,2.020306,102.081%,2.080886
530,Kick Sauber,2.253472,102.321%,2.321044


In [23]:
for index, row in transformed_laps_relative_avg_all.iterrows():
  if transformed_laps_relative_avg_all.loc[index, 'Delta_Avg_All']==0.0:
    transformed_laps_relative_avg_all.loc[index, 'Delta_Avg_All']=0.001
for index, row in transformed_laps_relative_avg_all.iterrows():
  if transformed_laps_relative_avg_all.loc[index, 'Delta_Percent_Abs']==0.0:
    transformed_laps_relative_avg_all.loc[index, 'Delta_Percent_Abs']=0.001   
        
transformed_laps_relative_avg_all

,Team,Delta_Avg_All,Delta_Percent_Avg,Delta_Percent_Abs
511,Ferrari,0.001000,100.0%,0.001000
513,McLaren,0.378806,100.39%,0.390164
515,Mercedes,0.497917,100.513%,0.512847
512,Red Bull Racing,0.961167,100.99%,0.989988
517,Racing Bulls,1.175389,101.211%,1.210634
520,Aston Martin,1.179944,101.215%,1.215326
525,Williams,1.201631,101.238%,1.237663
526,Haas F1 Team,1.837583,101.893%,1.892685
528,Alpine,2.020306,102.081%,2.080886
530,Kick Sauber,2.253472,102.321%,2.321044


In [24]:
fig=px.bar(
    transformed_laps_relative_avg_all,
    x="Team",
    y="Delta_Percent_Abs",
    color='Team',
    template="xgridoff",
    title="<b>Team Pace (Average Lap Times) for the {} {}</b>".format(year,gp),
    text="Delta_Percent_Avg",
    height=700, width=1200,
color_discrete_map={
                 "Alpine": "#0093cc",
                 "Aston Martin": "#229971",
                 "Ferrari": "#E80020",
                 "Haas F1 Team": "#B6BABD",
                 "Kick Sauber": "#52e252",
                 "McLaren": "#FF8000",
                 "Mercedes": "#27F4D2",
                 "RB": "#6692FF",
                 "Racing Bulls": "#6692FF",
                 "Red Bull Racing": "#3671C6",
                 "Williams": "#64C4FF" ,
                 "Alfa Romeo":"#C92D4B",
                 "AlphaTauri":"#5E8FAA",
                 "Racing Point":"#F596C8",
                 "Renault":"#FFF500",
                 "Toro Rosso":"#469bff",
                 "Force India":"#F596C8",
                 "Sauber":"#9B0000"
                 }
)

fig.update_traces(textposition='outside')

fig.update_traces(opacity=0.85)

fig.update_traces( marker_line_color='white',marker_line_width=1.5)

fig.update_yaxes(tickangle = -90,tickprefix="<b>",ticksuffix ="</b><br>")

fig.update_xaxes(tickangle = -0,tickprefix="<b>",ticksuffix ="</b><br>")

fig.update_yaxes(range=[transformed_laps_relative_avg_all['Delta_Percent_Abs'].max()+0.7, 0])

# fig['layout']['yaxis']['autorange'] = "reversed"

fig.update_layout(
    xaxis={'side': 'top'},
    yaxis={'side': 'left'}
)

fig.update_xaxes(title_text='')      # Remove axis titles
fig.update_yaxes(title_text='Percentage%')

fig.update_xaxes(
        title_standoff = 75
)

for x,y in zip(transformed_laps_relative_avg_all.Team, transformed_laps_relative_avg_all.Delta_Percent_Abs):
  for png in (Path(Path.cwd()).parents[0].joinpath("F1_LOGOS").glob("*.png")):
    if str.split(str.split(str(png),".")[0],"/")[6]==x:
      image=str(png)
      fig.add_layout_image(
          x=x,
          y=y+0.35,
          source=Image.open(image),
          xref="x",
          yref="y",
          sizex=0.9,
          sizey=0.9,
          xanchor="center",
          yanchor="middle",
      )

fig.update_layout(
    title_x=0.5,
    margin=dict(l=100, r=5, t=110, b=30),
    hoverlabel=dict(
    bgcolor="white",
    font_size=20,
    font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=15)),
    xaxis = dict(tickfont = dict(size=15)),
    font=dict(
        family="PT Sans Narrow",
        size=16,
        color="Black"
    ),
    title_font_family="PT Sans Narrow"
)
fig.update_traces( marker_line_color='Black',marker_line_width=0.1, opacity=0.99)

fig.show()

In [25]:
fig.write_html("/Users/rdesh723/statpulse-html/plots/Sprint Race Team Pace/Average Lap Times/{}/{} {} Driver Average Lap Plot.html".format(year, year,gp),full_html=False, include_plotlyjs='cdn')

In [ ]:
#Multi GP Plot

In [12]:
#2024
# race_multi=['Austrian Grand Prix','Miami Grand Prix','Chinese Grand Prix']
#2023
# race_multi=['Austrian Grand Prix','Azerbaijan Grand Prix','Belgian Grand Prix','Qatar Grand Prix','São Paulo Grand Prix','United States Grand Prix']
#2022
# race_multi=['Austrian Grand Prix','Emilia Romagna Grand Prix','São Paulo Grand Prix']
#2021
race_multi=['British Grand Prix','Italian Grand Prix','São Paulo Grand Prix']

year_multi=2021
for i in race_multi:
    track2=track3[(track3["GP"]==i)&(track3["Year"]==year_multi)]
    year=track2['Year'].iloc[0]
    gp=track2['GP'].iloc[0]
    track2['LapTime']= pd.to_timedelta(track2["LapTime"])
    track2=track2[track2['LapNumber']!=1.0]
    quicklaps=track2[track2["LapTime"]<track2['LapTime'].min()*1.07]
    transformed_laps = quicklaps.copy()
    transformed_laps.loc[:, "LapTime (s)"] = quicklaps["LapTime"].dt.total_seconds()

    # order the team from the fastest (lowest median lap time) tp slower
    team_order = (
        transformed_laps[["Team", "LapTime (s)"]].groupby("Team").median()["LapTime (s)"].sort_values().index
    )
    print(team_order)
    transformed_laps['median'] = transformed_laps['LapTime (s)'].groupby(transformed_laps['Team']).transform('median')
    transformed_laps.tail()
    fig_pace=px.box(
        transformed_laps.sort_values(by=["median","LapNumber"]),
        x="Team",
        y="LapTime (s)",
        color='Team',
        template="xgridoff",
        title="<b>Team Pace Plot for the {} {} Sprint Race</b>".format(year_multi,i),
        height=700, width=1200,

    color_discrete_map={
                 "Alpine": "#0093cc",
                 "Aston Martin": "#229971",
                 "Ferrari": "#E80020",
                 "Haas F1 Team": "#B6BABD",
                 "Kick Sauber": "#52e252",
                 "McLaren": "#FF8000",
                 "Mercedes": "#27F4D2",
                 "RB": "#6692FF",
                 "Red Bull Racing": "#3671C6",
                 "Williams": "#64C4FF" ,
                 "Alfa Romeo":"#C92D4B",
                 "Alfa Romeo Racing":"#C92D4B",
                 "AlphaTauri":"#5E8FAA",
                 "Racing Point":"#F596C8",
                 "Renault":"#FFF500",
                 "Toro Rosso":"#469bff",
                 "Force India":"#F596C8",
                 "Sauber":"#9B0000"
            }
    )
    fig_pace.update_traces(opacity=1)
    fig_pace.update_layout(
        title_x=0.5,
        hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
        ),
        yaxis = dict(tickfont = dict(size=15)),
        xaxis = dict(tickfont = dict(size=15)),
        font=dict(
            family="PT Sans Narrow",
            size=14,
            color="Black"
        ),
        title_font_family="PT Sans Narrow",
        margin=dict(l=50, r=5, t=35, b=50)
    )
    fig_pace.write_html("/Users/rdesh723/statpulse-html/plots/Sprint Race Team Pace/Team Pace Plots/{}/{} {} Sprint Race Team Pace Plot.html".format(year_multi,year_multi,i),full_html=False, include_plotlyjs='cdn')
    #Get MINIMUM OF THE MEDIAN LAP TIMES

    fastest_lap = transformed_laps["median"].min()
    transformed_laps['Delta']= transformed_laps["median"] - (fastest_lap)
    transformed_laps['Delta_Percent']= (transformed_laps["median"]/(fastest_lap)-1)*100
    transformed_laps['Delta_Percent']=transformed_laps['Delta_Percent']+100
    transformed_laps['Delta_Percent']=transformed_laps['Delta_Percent'].round(3)
    transformed_laps['Delta_Percent']=transformed_laps['Delta_Percent'].astype(str)
    transformed_laps['Delta_Percent']=transformed_laps['Delta_Percent']+"%"
    transformed_laps_relative_median=transformed_laps[["Team","Delta","Delta_Percent"]].drop_duplicates()
    transformed_laps_relative_median=transformed_laps_relative_median.sort_values("Delta")
    transformed_laps_relative_median
    for index, row in transformed_laps_relative_median.iterrows():
        if transformed_laps_relative_median.loc[index, 'Delta']==0.0:
         transformed_laps_relative_median.loc[index, 'Delta']=0.001
    transformed_laps_relative_median
    fig_median=px.bar(
        transformed_laps_relative_median,
        x="Team",
        y="Delta",
        color='Team',
        template="xgridoff",
        title="<b>Team Pace (Median Lap Times) for the {} {} Sprint Race</b>".format(year_multi,i),
        text="Delta_Percent",
        height=700, 
        width=1200,
        color_discrete_map={
                 "Alpine": "#0093cc",
                 "Aston Martin": "#229971",
                 "Ferrari": "#E80020",
                 "Haas F1 Team": "#B6BABD",
                 "Kick Sauber": "#52e252",
                 "McLaren": "#FF8000",
                 "Mercedes": "#27F4D2",
                 "RB": "#6692FF",
                 "Red Bull Racing": "#3671C6",
                 "Williams": "#64C4FF" ,
                 "Alfa Romeo":"#C92D4B",
                 "Alfa Romeo Racing":"#C92D4B",
                 "AlphaTauri":"#5E8FAA",
                 "Racing Point":"#F596C8",
                 "Renault":"#FFF500",
                 "Toro Rosso":"#469bff",
                 "Force India":"#F596C8",
                 "Sauber":"#9B0000"
            }
    )

    fig_median.update_traces(textposition='outside')

    fig_median.update_traces(opacity=0.85)

    fig_median.update_traces( marker_line_color='white',marker_line_width=1.5)

    fig_median.update_yaxes(tickangle = -90,tickprefix="<b>",ticksuffix ="</b><br>")

    fig_median.update_xaxes(tickangle = -0,tickprefix="<b>",ticksuffix ="</b><br>")

    fig_median.update_yaxes(range=[transformed_laps_relative_median['Delta'].max()+0.6, 0])

    fig_median.update_layout(
        xaxis={'side': 'top'},
        yaxis={'side': 'left'}
    )

    fig_median.update_xaxes(title_text='')      # Remove axis titles
    fig_median.update_yaxes(title_text='')

    fig_median.update_xaxes(
            title_standoff = 75
    )

    fig_median.update_layout(
        title_x=0.5,
        margin=dict(l=60, r=5, t=110, b=30),
        hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
        ),
        yaxis = dict(tickfont = dict(size=15)),
        xaxis = dict(tickfont = dict(size=15)),
        font=dict(
            family="PT Sans Narrow",
            size=12,
            color="Black"
        ),
        title_font_family="PT Sans Narrow"
    )
    fig_median.update_traces( marker_line_color='Black',marker_line_width=0.1, opacity=0.99)
    fig_median.write_html("/Users/rdesh723/statpulse-html/plots/Sprint Race Team Pace/Median Lap Times/{}/{} {} Sprint Race Team Median Plot.html".format(year_multi,year_multi,i),full_html=False, include_plotlyjs='cdn')
    
  #Get MINIMUM OF ALL LAP TIMES
    transformed_laps['min_all'] = transformed_laps['LapTime (s)'].groupby(transformed_laps['Team']).transform('min')
    fastest_lap_all = transformed_laps["min_all"].min()
    transformed_laps['Delta_Min_All']= transformed_laps["min_all"] - (fastest_lap_all)
    transformed_laps['Delta_Percent_All']= (transformed_laps["min_all"]/(fastest_lap_all)-1)*100
    transformed_laps['Delta_Percent_All']=transformed_laps['Delta_Percent_All']+100
    transformed_laps['Delta_Percent_All']=transformed_laps['Delta_Percent_All'].round(3)
    transformed_laps['Delta_Percent_All']=transformed_laps['Delta_Percent_All'].astype(str)
    transformed_laps['Delta_Percent_All']=transformed_laps['Delta_Percent_All']+"%"
    transformed_laps_relative_min_all=transformed_laps[["Team","Delta_Min_All","Delta_Percent_All"]].drop_duplicates()
    transformed_laps_relative_min_all=transformed_laps_relative_min_all.sort_values("Delta_Min_All")
    transformed_laps_relative_min_all
    for index, row in transformed_laps_relative_min_all.iterrows():
        if transformed_laps_relative_min_all.loc[index, 'Delta_Min_All']==0.0:
            transformed_laps_relative_min_all.loc[index, 'Delta_Min_All']=0.001
    fig_min=px.bar(
        transformed_laps_relative_min_all,
        x="Team",
        y="Delta_Min_All",
        color='Team',
        template="xgridoff",
        title="<b>Team Pace (Fastest Lap Times) for the {} {} Sprint Race</b>".format(year,gp),
        text="Delta_Percent_All",
        height=700, width=1200,
    color_discrete_map={
                     "Alpine": "#0093cc",
                 "Aston Martin": "#229971",
                 "Ferrari": "#E80020",
                 "Haas F1 Team": "#B6BABD",
                 "Kick Sauber": "#52e252",
                 "McLaren": "#FF8000",
                 "Mercedes": "#27F4D2",
                 "RB": "#6692FF",
                 "Red Bull Racing": "#3671C6",
                 "Williams": "#64C4FF" ,
                 "Alfa Romeo":"#C92D4B",
                 "Alfa Romeo Racing":"#C92D4B",
                 "AlphaTauri":"#5E8FAA",
                 "Racing Point":"#F596C8",
                 "Renault":"#FFF500",
                 "Toro Rosso":"#469bff",
                 "Force India":"#F596C8",
                 "Sauber":"#9B0000"
                    }
    )

    fig_min.update_traces(textposition='outside')

    fig_min.update_traces(opacity=0.85)

    fig_min.update_traces( marker_line_color='white',marker_line_width=1.5)

    fig_min.update_yaxes(tickangle = -90,tickprefix="<b>",ticksuffix ="</b><br>")

    fig_min.update_xaxes(tickangle = -0,tickprefix="<b>",ticksuffix ="</b><br>")

    fig_min.update_yaxes(range=[transformed_laps_relative_min_all['Delta_Min_All'].max()+0.5, 0])

    # fig['layout']['yaxis']['autorange'] = "reversed"

    fig_min.update_layout(
        xaxis={'side': 'top'},
        yaxis={'side': 'left'}
    )

    fig_min.update_xaxes(title_text='')      # Remove axis titles
    fig_min.update_yaxes(title_text='')

    fig_min.update_xaxes(
            title_standoff = 75
    )

    fig_min.update_layout(
        title_x=0.5,
        margin=dict(l=60, r=5, t=110, b=30),
        hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
        ),
        yaxis = dict(tickfont = dict(size=15)),
        xaxis = dict(tickfont = dict(size=15)),
        font=dict(
            family="PT Sans Narrow",
            size=14,
            color="Black"
        ),
        title_font_family="PT Sans Narrow"
    )
    fig_min.update_yaxes(ticksuffix = "  ")
    fig_min.update_traces( marker_line_color='Black',marker_line_width=0.1, opacity=0.99)
    fig_min.write_html("/Users/rdesh723/statpulse-html/plots/Sprint Race Team Pace/Fastest Lap/{}/{} {} Sprint Race Team Fastest Lap Plot.html".format(year_multi,year_multi,i),full_html=False, include_plotlyjs='cdn')

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_12111/2708169292.py:15: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_12111/2708169292.py:15: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_12111/2708169292.py:15: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th

Index(['Mercedes', 'Red Bull Racing', 'McLaren', 'Ferrari', 'Aston Martin',
       'AlphaTauri', 'Alpine', 'Williams', 'Alfa Romeo Racing',
       'Haas F1 Team'],
      dtype='object', name='Team')
Index(['Mercedes', 'Red Bull Racing', 'McLaren', 'Ferrari', 'Aston Martin',
       'Alpine', 'Williams', 'AlphaTauri', 'Alfa Romeo Racing',
       'Haas F1 Team'],
      dtype='object', name='Team')
Index(['Mercedes', 'Red Bull Racing', 'Ferrari', 'McLaren', 'Alpine',
       'AlphaTauri', 'Aston Martin', 'Alfa Romeo Racing', 'Williams',
       'Haas F1 Team'],
      dtype='object', name='Team')
